# 00 · Panorama de la plataforma MLOps

Este notebook valida, desde JupyterLab, la conectividad con PostgreSQL y las capas Bronze/Silver/Gold, MLflow, MinIO y los servicios internos de K3s. Es evidencia de integración; no modifica datos ni modelos.

In [1]:
import os
import boto3
import mlflow
import pandas as pd
import requests
from sqlalchemy import create_engine
from IPython.display import display

# PostgreSQL: capas Medallion
db_url = (
    f"postgresql+psycopg2://{os.environ['POSTGRES_USER']}:"
    f"{os.environ['POSTGRES_PASSWORD']}@postgres:5432/"
    f"{os.environ['POSTGRES_DB']}"
)
engine = create_engine(db_url)

layers = pd.read_sql("""
    SELECT
        schemaname AS capa,
        relname AS tabla,
        n_live_tup AS filas_estimadas
    FROM pg_stat_user_tables
    WHERE schemaname IN ('raw', 'clean', 'gold', 'mlops')
    ORDER BY
        CASE schemaname
            WHEN 'raw' THEN 1
            WHEN 'clean' THEN 2
            WHEN 'gold' THEN 3
            WHEN 'mlops' THEN 4
        END,
        relname
""", engine)

# MLflow
mlflow.set_tracking_uri("http://mlflow:5000")
experiments = pd.DataFrame(
    [(e.experiment_id, e.name) for e in mlflow.search_experiments()],
    columns=["id_experimento", "experimento"],
)

# MinIO mediante boto3/S3
s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["MLFLOW_S3_ENDPOINT_URL"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
)
buckets = pd.DataFrame(
    [{"bucket": b["Name"]} for b in s3.list_buckets()["Buckets"]]
)

# Servicios accesibles desde la red interna de K3s
urls = {
    "Data API": "http://data-api:80/health",
    "Property API": "http://property-api:8000/health",
    "MLflow": "http://mlflow:5000/health",
    "Prometheus": "http://prometheus:9090/-/ready",
    "Grafana": "http://grafana:3000/api/health",
    "Streamlit": "http://streamlit:8501/_stcore/health",
}

service_status = []
for service, url in urls.items():
    try:
        response = requests.get(url, timeout=5)
        service_status.append(
            {"servicio": service, "codigo_http": response.status_code, "estado": "conectado"}
        )
    except Exception as exc:
        service_status.append(
            {"servicio": service, "codigo_http": None, "estado": f"error: {type(exc).__name__}"}
        )

print("Capas y tablas Medallion")
display(layers)

print("Experimentos MLflow")
display(experiments)

print("Buckets MinIO")
display(buckets)

print("Conectividad interna desde Jupyter")
display(pd.DataFrame(service_status))

Capas y tablas Medallion


,capa,tabla,filas_estimadas
0,raw,patient_records,0
1,raw,property_batches,6
2,clean,patient_features,0
3,clean,property_records,682156
4,gold,property_training_features,682153
5,mlops,batch_audit,5
6,mlops,inference_logs,574
7,mlops,preprocessing_audit,0
8,mlops,training_splits,0


Experimentos MLflow


,id_experimento,experimento
0,3,property_price_regression
1,2,diabetes_readmission
2,1,infrastructure_smoke_test
3,0,Default


Buckets MinIO


,bucket
0,mlflow


Conectividad interna desde Jupyter


,servicio,codigo_http,estado
0,Data API,200,conectado
1,Property API,200,conectado
2,MLflow,200,conectado
3,Prometheus,200,conectado
4,Grafana,200,conectado
5,Streamlit,200,conectado


In [2]:
raw_property_batches = pd.read_sql("""
    SELECT
        batch_id,
        ingestion_mode,
        record_count,
        retrieved_at,
        LEFT(payload, 400) AS vista_previa_payload
    FROM raw.property_batches
    ORDER BY retrieved_at
""", engine)

raw_property_batches

,batch_id,ingestion_mode,record_count,retrieved_at,vista_previa_payload
0,group-5-batch-0,manual,73784,2026-11-15 16:56:33.597503+00:00,"{""group_number"":5,""batch_number"":0,""data"":[{""b..."
1,group-5-batch-1,airflow,94551,2026-11-15 18:02:56.889196+00:00,"{""group_number"":5,""batch_number"":1,""data"":[{""b..."
2,group-5-batch-2,airflow,230366,2026-11-15 20:43:25.550203+00:00,"{""group_number"":5,""batch_number"":2,""data"":[{""b..."
3,group-5-batch-3,airflow,4055,2026-11-15 21:18:39.545288+00:00,"{""group_number"":5,""batch_number"":3,""data"":[{""b..."
4,group-5-batch-4,airflow,320696,2026-11-15 21:29:44.869631+00:00,"{""group_number"":5,""batch_number"":4,""data"":[{""b..."
5,group-5-batch-5,airflow,361457,2026-11-15 23:00:33.641442+00:00,"{""group_number"":5,""batch_number"":5,""data"":[{""b..."


In [3]:
silver_property_records = pd.read_sql("""
    SELECT *
    FROM clean.property_records
    LIMIT 10
""", engine)

silver_property_records

,id,batch_id,source_record_index,row_hash,features,target_price,transformed_at
0,4056,group-5-batch-4,0,c7271aeed922a08e6c8ccbbd294baca8622c07bf1d7ef3...,"{'bed': 3.0, 'bath': 2.0, 'city': 'Agawam', 's...",384900.0,2026-11-15 21:29:49.012971+00:00
1,4057,group-5-batch-4,1,51d7790958abada5e1f525846e42743303304f451905cf...,"{'bed': 4.0, 'bath': 2.0, 'city': 'Pelham', 's...",419000.0,2026-11-15 21:29:49.012971+00:00
2,4058,group-5-batch-4,2,7a3ab4f51d42eeecf2abbce8f412f901cadd4ea0499363...,"{'bed': 4.0, 'bath': 2.0, 'city': 'Amherst', '...",415000.0,2026-11-15 21:29:49.012971+00:00
3,4059,group-5-batch-4,3,ca3b0f4272b9c7d8f76e62cacf4341d1c17ebde44444bc...,"{'bed': 3.0, 'bath': 2.0, 'city': 'Amherst', '...",390000.0,2026-11-15 21:29:49.012971+00:00
4,4060,group-5-batch-4,4,9a33f7006978185640c154cc764e46d346c2a931a6113b...,"{'bed': 4.0, 'bath': 3.0, 'city': 'Amherst', '...",499900.0,2026-11-15 21:29:49.012971+00:00
5,4061,group-5-batch-4,5,e0ae8d0c7fe7ed9fdad3d1a8ce9a7295df6645ff40aac5...,"{'bed': 3.0, 'bath': 2.0, 'city': 'Amherst', '...",439900.0,2026-11-15 21:29:49.012971+00:00
6,4062,group-5-batch-4,6,6daa48e9f06019937361a4e8ab8e5948aa61afecafb7b2...,"{'bed': 2.0, 'bath': 2.0, 'city': 'Granby', 's...",397000.0,2026-11-15 21:29:49.012971+00:00
7,4063,group-5-batch-4,7,d447fed46dedfc46c8aaad5fb7746d0a3aefd099de46d9...,"{'bed': 2.0, 'bath': 2.0, 'city': 'Whately', '...",389900.0,2026-11-15 21:29:49.012971+00:00
8,4064,group-5-batch-4,8,6787b1cfcec0d9077cedfb80adfe89999340cf71045993...,"{'bed': 4.0, 'bath': 3.0, 'city': 'Sunderland'...",459900.0,2026-11-15 21:29:49.012971+00:00
9,4065,group-5-batch-4,9,a0877f2f13290a859274151bf04dbe28fbd61081688ee0...,"{'bed': 4.0, 'bath': 2.0, 'city': 'Granby', 's...",324900.0,2026-11-15 21:29:49.012971+00:00


In [4]:
gold_training_features = pd.read_sql("""
    SELECT *
    FROM gold.property_training_features
    LIMIT 10
""", engine)

gold_training_features

,clean_id,batch_id,target_price,brokered_by,status,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_year,prepared_at
0,78719,group-5-batch-4,448900.0,52777.0,for_sale,3.0,2.0,0.26,876056.0,Lonsdale,Minnesota,55046.0,3330.0,2004,2026-11-15 21:30:31.878189+00:00
1,78720,group-5-batch-4,356900.0,57847.0,for_sale,3.0,2.0,0.16,831243.0,Lonsdale,Minnesota,55046.0,2895.0,2018,2026-11-15 21:30:31.878189+00:00
2,78721,group-5-batch-4,354900.0,57847.0,for_sale,3.0,2.0,0.16,932105.0,Lonsdale,Minnesota,55046.0,2895.0,2018,2026-11-15 21:30:31.878189+00:00
3,78722,group-5-batch-4,356900.0,57847.0,for_sale,3.0,2.0,0.16,795398.0,Lonsdale,Minnesota,55046.0,2895.0,2017,2026-11-15 21:30:31.878189+00:00
4,78723,group-5-batch-4,449950.0,12874.0,for_sale,3.0,2.0,0.32,717193.0,Madison Lake,Minnesota,56063.0,1807.0,2022,2026-11-15 21:30:31.878189+00:00
5,78724,group-5-batch-4,392708.0,52777.0,for_sale,3.0,2.0,0.29,279099.0,Montgomery,Minnesota,56069.0,2800.0,2013,2026-11-15 21:30:31.878189+00:00
6,78725,group-5-batch-4,324500.0,10209.0,for_sale,2.0,3.0,0.03,13308.0,Owatonna,Minnesota,55060.0,1862.0,2019,2026-11-15 21:30:31.878189+00:00
7,78726,group-5-batch-4,419000.0,81491.0,for_sale,3.0,3.0,0.48,1099370.0,Waseca,Minnesota,56093.0,3648.0,2020,2026-11-15 21:30:31.878189+00:00
8,78727,group-5-batch-4,339900.0,81071.0,for_sale,3.0,2.0,1.90,1318173.0,Waterville,Minnesota,56096.0,2030.0,1994,2026-11-15 21:30:31.878189+00:00
9,78728,group-5-batch-4,424900.0,78799.0,for_sale,4.0,3.0,0.45,128384.0,Waseca,Minnesota,56093.0,3615.0,2022,2026-11-15 21:30:31.878189+00:00
